huggingface pipline

In [8]:
import os
from dotenv import load_dotenv

hf_token = os.getenv("HF_TOKEN")

In [10]:
from transformers import pipeline

# 1. 텍스트 생성 파이프라인 불러오기
# search_optimizer = pipeline("text-generation", model="google/gemma-2b-it", token=hf_token)
# classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", token=hf_token)
optmizer_model = "microsoft/Phi-4-mini-instruct"
# optmizer_model = "skt/kogpt2-base-v2"
# optmizer_model = "Jackrong/Qwen3.5-4B-Claude-4.6-Opus-Reasoning-Distilled-GGUF"
# optmizer_model = "CohereLabs/tiny-aya-global"
# optmizer_model = "CohereLabs/c4ai-command-r-plus"

class_model = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"

classifier = pipeline("zero-shot-classification", model=class_model, token=hf_token)
search_optimizer = pipeline("text-generation", model=optmizer_model, token=hf_token)


Device set to use cpu


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cpu


In [16]:
import re
def optimize_query(user_input):
    system_prompt = """ 
        
        너는 검색어 전문가야. 
        내가 궁금한 내용을 구어체로 너에게 줄게.
        너는 내가 궁금한 내용이 검색 결과로 나올 수 있도록 추천 포털 검색어를 생성해 줘.
        검색어는 핵심만 포함하여 간결하게.
        다른 글자는 없이 오직 완성된 검색어만 출력해.

        """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"내가 찾아볼 후기: ---- \n '{user_input}'"}
    ]
    
    # 모델 출력 생성
    # return_full_text=False를 통해 프롬프트를 제외한 답변만 추출
    outputs = search_optimizer(messages, max_new_tokens=30, do_sample=False, return_full_text=False)
    
    # 결과 정제
    query = outputs[0]['generated_text'].strip()
    
    # 혹시 모를 따옴표나 불필요한 서술어 제거 (포스트 프로세싱)
    query = re.sub(r'[^a-zA-Z0-9가-힣\s]', '', query)
    return query

def classify_intent(user_msg):
    # 2. 사용자의 질문이 어떤 검색 의도인지 분류 (Zero-shot)
    # 별도의 학습 데이터 없이 아래 'candidate_labels'만으로 분류합니다.
    candidate_labels = ["IT 및 전자기기", "패션 및 뷰티", "맛집 및 카페", "여행 및 숙박", "생활 및 의료", "공연 및 문화", "경제 및 정치", "쇼핑", "사전 및 정보"]
    
    result = classifier(user_msg, candidate_labels)
    top_intent = result['labels'][0] # 가장 확률이 높은 의도 추출
    
    return top_intent


In [17]:

# 실행 테스트
# user_input = input("검색할 내용: ")
user_input = "내가 강남에서 1시간 정도 시간이 남아. 친구 2명과 함께인데 뭘 하면 좋을까?"

print("--- Hugging Face Pipeline 실행 중 ---")
intent = classify_intent(user_input)
query = optimize_query(user_input)

print(f"입력 문장: {user_input}")
print(f"분석된 의도: {intent}")
print(f"최종 추천 검색어: {query}")


--- Hugging Face Pipeline 실행 중 ---
입력 문장: 내가 강남에서 1시간 정도 시간이 남아. 친구 2명과 함께인데 뭘 하면 좋을까?
분석된 의도: 여행 및 숙박
최종 추천 검색어: 강남에서 친구와 함께 1시간 남아 있는 경우 추천 강남 친구 활동 1시간
